In [1]:
import pandas as pd

data_path = "data/raw/"

df = pd.read_csv(data_path + "salaries_concat.csv")

df.head()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2021,MI,FT,Data Scientist,30400000,CLP,40038,CL,100.0,CL,L
1,2021,MI,FT,BI Data Analyst,11000000,HUF,36259,HU,50.0,US,L
2,2020,MI,FT,Data Scientist,11000000,HUF,35735,HU,50.0,HU,L
3,2021,MI,FT,ML Engineer,8500000,JPY,77364,JP,50.0,JP,S
4,2022,SE,FT,Lead Machine Learning Engineer,7500000,INR,95386,IN,50.0,IN,L


In [2]:
# imputar remote_ratio
df["remote_ratio"] = df["remote_ratio"].fillna("Unknown")


# ---- MAPEO EXPERIENCE LEVEL ----
experience_map = {
    "Entry": "EN",
    "Junior": "EN",
    "Mid": "MI",
    "Senior": "SE",
    "Expert": "EX"
}

df["experience_level"] = df["experience_level"].replace(experience_map)


# ---- MAPEO EMPLOYMENT TYPE ----
employment_map = {
    "Full-Time": "FT",
    "Part-Time": "PT",
    "Contract": "CT",
    "Freelance": "FL"
}

df["employment_type"] = df["employment_type"].replace(employment_map)


# ---- MAPEO COMPANY SIZE ----
company_size_map = {
    "Small": "S",
    "Medium": "M",
    "Large": "L"
}

df["company_size"] = df["company_size"].replace(company_size_map)

# ---- MAPEO COMPANY LOCATION Y EMPLOYEE RESIDENCE ----

country_map = {
    "Algeria": "DZ",
    "American Samoa": "AS",
    "Andorra": "AD",
    "Argentina": "AR",
    "Armenia": "AM",
    "Bahamas": "BS",
    "Belgium": "BE",
    "Bosnia and Herzegovina": "BA",
    "Brazil": "BR",
    "Central African Republic": "CF",
    "Chile": "CL",
    "China": "CN",
    "Colombia": "CO",
    "Croatia": "HR",
    "Czechia": "CZ",
    "Denmark": "DK",
    "Ecuador": "EC",
    "Egypt": "EG",
    "Estonia": "EE",
    "Finland": "FI",
    "France": "FR",
    "Ghana": "GH",
    "Greece": "GR",
    "Honduras": "HN",
    "Hong Kong": "HK",
    "Hungary": "HU",
    "Indonesia": "ID",
    "Iran, Islamic Republic of": "IR",
    "Iraq": "IQ",
    "Israel": "IL",
    "Kenya": "KE",
    "Korea, Republic of": "KR",
    "Latvia": "LV",
    "Lithuania": "LT",
    "Luxembourg": "LU",
    "Malaysia": "MY",
    "Malta": "MT",
    "Mexico": "MX",
    "Moldova, Republic of": "MD",
    "New Zealand": "NZ",
    "Nigeria": "NG",
    "Norway": "NO",
    "Pakistan": "PK",
    "Philippines": "PH",
    "Puerto Rico": "PR",
    "Romania": "RO",
    "Russian Federation": "RU",
    "Saudi Arabia": "SA",
    "Slovenia": "SI",
    "South Africa": "ZA",
    "Sweden": "SE",
    "Thailand": "TH",
    "Turkey": "TR",
    "Ukraine": "UA",
    "United Arab Emirates": "AE"
}

df["company_location"] = df["company_location"].replace(country_map)
df["employee_residence"] = df["employee_residence"].replace(country_map)

# ---- MAPEO REMOTE RATIO ----

remote_map = {
    0: "On-site",
    50: "Hybrid",
    100: "Remote",
    "Unknown": "Unknown"
}

df["remote_type"] = df["remote_ratio"].map(remote_map)


# limpiar espacios en job_title
df["job_title"] = df["job_title"].str.strip()

In [3]:
df = df.drop_duplicates(
    subset=[
        "work_year",
        "job_title",
        "salary_in_usd",
        "employee_residence",
        "company_location"
    ]
)

In [4]:
print("Nuevo tamaño dataset:", df.shape)

Nuevo tamaño dataset: (10335, 12)


In [5]:
df["job_title"].unique()

array(['Data Scientist', 'BI Data Analyst', 'ML Engineer',
       'Lead Machine Learning Engineer', 'Data Science Manager',
       'Head of Machine Learning', 'Research Engineer',
       'Head of Data Science', 'AI Programmer',
       'Machine Learning Engineer', 'Lead Data Scientist',
       'Data Engineer', 'Applied Machine Learning Scientist',
       'Lead Data Analyst', 'Data Analytics Manager',
       'Data Integration Specialist', 'Principal Data Architect',
       'NLP Engineer', 'Big Data Engineer', 'AI Research Engineer',
       'Machine Learning Software Engineer', 'Data Analyst',
       'Applied Data Scientist', 'AI Scientist', 'Data Analytics Lead',
       'Business Data Analyst', 'Product Data Analyst',
       'Computer Vision Engineer', 'Data Science Consultant',
       'AI Architect', 'Analytics Engineer', 'Machine Learning Scientist',
       'Research Scientist', 'Prompt Engineer',
       'Principal Data Scientist', 'Applied Scientist',
       'Deep Learning Engineer', 

In [6]:
def categorize_role(title):

    title = title.lower()

    if "research" in title:
        return "Research"

    elif "analyst" in title or "analytics" in title or "bi" in title or "business intelligence" in title:
        return "Data Analyst / BI"

    elif "machine learning" in title or "ml" in title or "ai" in title or "deep learning" in title or "computer vision" in title or "nlp" in title:
        return "ML / AI Engineer"

    elif "scientist" in title or "data science" in title:
        return "Data Scientist"

    elif "engineer" in title:
        return "Data Engineer"

    elif "architect" in title or "etl" in title or "integration" in title or "modeler" in title or "modeller" in title or "developer" in title:
        return "Data Architecture / Infrastructure"

    elif "manager" in title or "head" in title or "director" in title or "lead" in title or "strategist" in title or "product owner" in title:
        return "Management"
    
    elif "specialist" in title and "data" in title:
        return "Data Analyst / BI"

    elif "operations" in title:
        return "Data Engineer"

    elif "management" in title:
        return "Data Architecture / Infrastructure"

    elif "autonomous" in title:
        return "ML / AI Engineer"

    else:
        return "Other"

In [7]:
df["role_category"] = df["job_title"].apply(categorize_role)

In [8]:
df["role_category"].value_counts()

role_category
Data Analyst / BI                     2622
Data Scientist                        2588
Data Engineer                         1995
ML / AI Engineer                      1787
Research                               672
Data Architecture / Infrastructure     413
Management                             258
Name: count, dtype: int64

In [9]:
df[df["role_category"] == "Other"]["job_title"].value_counts().head(20)

Series([], Name: count, dtype: int64)

In [10]:
df["salary_band"] = pd.qcut(
    df["salary_in_usd"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)

In [11]:
df["international_job"] = (
    df["company_location"] != df["employee_residence"]
)

In [12]:
df.head()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,remote_type,role_category,salary_band,international_job
0,2021,MI,FT,Data Scientist,30400000,CLP,40038,CL,100.0,CL,L,Remote,Data Scientist,Low,False
1,2021,MI,FT,BI Data Analyst,11000000,HUF,36259,HU,50.0,US,L,Hybrid,Data Analyst / BI,Low,True
2,2020,MI,FT,Data Scientist,11000000,HUF,35735,HU,50.0,HU,L,Hybrid,Data Scientist,Low,False
3,2021,MI,FT,ML Engineer,8500000,JPY,77364,JP,50.0,JP,S,Hybrid,ML / AI Engineer,Low,False
4,2022,SE,FT,Lead Machine Learning Engineer,7500000,INR,95386,IN,50.0,IN,L,Hybrid,ML / AI Engineer,Medium,False


In [13]:
df.to_csv("data/processed/salaries_clean.csv", index=False)